In [ ]:
import numpy as np
from collections import defaultdict
from typing import Dict, List, Set, Tuple
import time
import math

class EWCA:
    def __init__(self, label_id: Dict[str, int], id_label: Dict[int, str], relations: Dict[int, List[int]]):
        
        self.label_id = label_id
        self.id_label = id_label
        self.relations = relations
        self.N = len(label_id)
        self.structural_similarity_threshold = 0.4
        
        # Initialize weight matrices
        self.W1 = np.zeros((self.N, self.N))
        self.Ws = np.zeros((self.N, self.N))
    
    def calculate_jaccard_distance(self) -> None:
        
        for id1 in self.relations:
            neighbors1 = set(self.relations[id1])
            for id2 in self.relations[id1]:
                if id2 > id1:  # Process each pair only once
                    neighbors2 = set(self.relations[id2])
                    
                    if len(neighbors1) > 1 or len(neighbors2) > 1:
                        intersection = neighbors1 & neighbors2
                        union = neighbors1 | neighbors2
                        
                        if len(intersection) == 0:
                            weight = 0.0
                        else:
                            weight = len(intersection) / len(union)
                            
                        self.W1[id1, id2] = weight
                        self.W1[id2, id1] = weight
                    else:
                        self.W1[id1, id2] = 0.0
                        self.W1[id2, id1] = 0.0
    
    def calculate_ecv2_weights(self) -> None:
        
        for id1 in self.relations:
            neighbors1 = set(self.relations[id1])
            for id2 in self.relations[id1]:
                if id1 < id2:
                    neighbors2 = set(self.relations[id2])
                    common_neighbors = neighbors1 & neighbors2
                    
                    if len(common_neighbors) > 0:
                        sum_common_weight = sum(
                            self.W1[id1, iw] * self.W1[id2, iw] 
                            for iw in common_neighbors
                        )
                        h_second = sum_common_weight
                        h_second1 = (h_second + self.W1[id1, id2]) / (len(common_neighbors) + 1)
                        
                        self.Ws[id1, id2] = h_second1
                        self.Ws[id2, id1] = h_second1
        
        # Filter interactions based on weights
        self.filter_interactions()
    
    def filter_interactions(self) -> None:
        
        # Create a new relations dictionary with filtered interactions
        new_relations = defaultdict(list)
        
        for id1 in self.relations:
            for id2 in self.relations[id1]:
                if self.Ws[id1, id2] > 0.0:
                    new_relations[id1].append(id2)
        
        self.relations = {
            k: v for k, v in new_relations.items() 
            if len(v) > 1
        }
    
    def calculate_structural_similarity(self, id1: int, id2: int) -> float:
       
        neighbors1 = set(self.relations.get(id1, []))
        neighbors2 = set(self.relations.get(id2, []))
        
        # Include the proteins themselves in their neighborhoods
        neighbors1.add(id1)
        neighbors2.add(id2)
        
        common_neighbors = neighbors1 & neighbors2
        denominator = math.sqrt(len(neighbors1) * len(neighbors2))
        
        return len(common_neighbors) / denominator if denominator > 0 else 0.0
    
    def detect_core_complexes(self) -> Dict[int, List[int]]:
        
        core_complexes = {}
        
        for id1 in self.relations:
            complex_members = {id1}
            
            for id2 in self.relations[id1]:
                if id1 < id2:
                    similarity = self.calculate_structural_similarity(id1, id2)
                    if similarity > self.structural_similarity_threshold:
                        complex_members.add(id2)
            
            if len(complex_members) >= 2:
                core_complexes[id1] = list(complex_members)
        
        return core_complexes
    
    def find_attachments(self, core_complexes: Dict[int, List[int]]) -> Dict[int, List[int]]:
    
        complexes = {}
        
        for core_id, core_members in core_complexes.items():
            # Find all neighboring proteins not in the core
            attachment_candidates = set()
            for protein in core_members:
                attachment_candidates.update(
                    p for p in self.relations.get(protein, []) 
                    if p not in core_members
                )
            
            if not attachment_candidates:
                complexes[core_id] = core_members
                continue
            
            # Calculate average edge weight within the core
            in_edges, _ = self.clustering_coefficient(core_members)
            avg_core_edges = 2 * in_edges / len(core_members)
            
            # Classify attachment candidates
            local_attachments = set()
            overlapping_attachments = set()
            
            for candidate in attachment_candidates:
                in_weight = 0.0
                out_weight = 0.0
                count = 0
                
                for neighbor in self.relations.get(candidate, []):
                    if neighbor in core_members:
                        in_weight += self.Ws[candidate, neighbor]
                        count += 1
                    else:
                        out_weight += self.Ws[candidate, neighbor]
                
                if count >= 2:
                    if in_weight <= out_weight:
                        if in_weight >= 0.5 * avg_core_edges:
                            overlapping_attachments.add(candidate)
                    else:
                        local_attachments.add(candidate)
            
            # Combine all attachments
            all_attachments = local_attachments | overlapping_attachments
            complexes[core_id] = core_members + list(all_attachments)
        
        return complexes
    
    def clustering_coefficient(self, proteins: List[int]) -> Tuple[float, int]:
        """
        Calculate clustering coefficient for a set of proteins.
        Returns (sum_of_in_edges, count_of_edges)
        """
        protein_set = set(proteins)
        sum_edges = 0.0
        count = 0
        
        for i, id1 in enumerate(proteins):
            for id2 in proteins[i+1:]:
                if id2 in self.relations.get(id1, []):
                    sum_edges += self.Ws[id1, id2]
                    count += 1
        
        return sum_edges, count
    
    def overlap_score(self, set1: Set[int], set2: Set[int]) -> float:
        
        intersection = set1 & set2
        union = set1 | set2
        return len(intersection) / len(union) if union else 0.0
    
    def filter_redundant_complexes(self, complexes: Dict[int, List[int]], 
                                 overlap_threshold: float = 0.8) -> Dict[int, List[int]]:
    
        visited = set()
        result = {}
        complex_items = list(complexes.items())
        
        for i, (id1, members1) in enumerate(complex_items):
            if id1 in visited:
                continue
                
            current_members = set(members1)
            
            for j, (id2, members2) in enumerate(complex_items[i+1:], i+1):
                if id2 in visited:
                    continue
                    
                score = self.overlap_score(current_members, set(members2))
                if score >= overlap_threshold:
                    visited.add(id2)
                    current_members.update(members2)
            
            result[id1] = list(current_members)
        
        return result
    
    def save_results(self, complexes: Dict[int, List[int]], output_file: str) -> None:
        
        with open(output_file, 'w') as f:
            for complex_id, members in complexes.items():
                if len(members) >= 3:  # Only save complexes with at least 3 proteins
                    proteins = [self.id_to_protein[pid] for pid in members]
                    line = f"{complex_id}\t{' '.join(proteins)}\n"
                    f.write(line)
    
    def run(self, ss_threshold: float = 0.4) -> Dict[int, List[int]]:
        self.structural_similarity_threshold = ss_threshold
        start_time = time.time()
        
        print("Calculating Jaccard distances...")
        self.calculate_jaccard_distance()
        
        print("Calculating ECV2 weights...")
        self.calculate_ecv2_weights()
        
        print("Detecting core complexes...")
        core_complexes = self.detect_core_complexes()
        
        print("Finding attachment proteins...")
        complexes = self.find_attachments(core_complexes)
        
        print("Filtering redundant complexes...")
        filtered_complexes = self.filter_redundant_complexes(complexes)
        
        elapsed = time.time() - start_time
        print(f"EWCA completed in {elapsed:.2f} seconds")
        
        return filtered_complexes
        




EWCA

In [3]:
import numpy as np
from collections import defaultdict
from typing import Dict, List, Set, Tuple
import time
import math

class EWCA:
    def __init__(self, interaction_file: str, structural_similarity_threshold: float = 0.4):
        
        self.interaction_file = interaction_file
        self.structural_similarity_threshold = structural_similarity_threshold
        
        # Data structures
        self.relations: Dict[int, List[int]] = defaultdict(list)
        self.protein_to_id: Dict[str, int] = {}
        self.id_to_protein: Dict[int, str] = {}
        self.N = 0  # Number of proteins
        
        # Weights
        self.W1: np.ndarray = None
        self.Ws: np.ndarray = None
        
    def load_interactions(self) -> None:
        
        total_interactions = 0
        
        with open(self.interaction_file, 'r') as f:
            for line in f:
                if line.strip() == "":
                    continue
                    
                parts = line.strip().split('\t')
                if len(parts) < 2:
                    continue
                    
                protein1, protein2 = parts[0], parts[1]
                
                # Assign IDs to proteins if not already assigned
                if protein1 not in self.protein_to_id:
                    self.protein_to_id[protein1] = self.N
                    self.id_to_protein[self.N] = protein1
                    self.N += 1
                    
                if protein2 not in self.protein_to_id:
                    self.protein_to_id[protein2] = self.N
                    self.id_to_protein[self.N] = protein2
                    self.N += 1
                    
                id1, id2 = self.protein_to_id[protein1], self.protein_to_id[protein2]
                
                # Add interaction (undirected)
                if id1 != id2 and id2 not in self.relations[id1]:
                    total_interactions += 1
                    self.relations[id1].append(id2)
                    self.relations[id2].append(id1)
        
        print(f"Total number of proteins: {self.N}")
        print(f"Total number of interactions: {total_interactions}")
        
        # Initialize weight matrices
        self.W1 = np.zeros((self.N, self.N))
        self.Ws = np.zeros((self.N, self.N))
    
    def calculate_jaccard_distance(self) -> None:
        
        for id1 in self.relations:
            neighbors1 = set(self.relations[id1])
            for id2 in self.relations[id1]:
                if id2 > id1:  # Process each pair only once
                    neighbors2 = set(self.relations[id2])
                    
                    if len(neighbors1) > 1 or len(neighbors2) > 1:
                        intersection = neighbors1 & neighbors2
                        union = neighbors1 | neighbors2
                        
                        if len(intersection) == 0:
                            weight = 0.0
                        else:
                            weight = len(intersection) / len(union)
                            
                        self.W1[id1, id2] = weight
                        self.W1[id2, id1] = weight
                    else:
                        self.W1[id1, id2] = 0.0
                        self.W1[id2, id1] = 0.0
    
    def calculate_ecv2_weights(self) -> None:
        
        for id1 in self.relations:
            neighbors1 = set(self.relations[id1])
            for id2 in self.relations[id1]:
                if id1 < id2:
                    neighbors2 = set(self.relations[id2])
                    common_neighbors = neighbors1 & neighbors2
                    
                    if len(common_neighbors) > 0:
                        sum_common_weight = sum(
                            self.W1[id1, iw] * self.W1[id2, iw] 
                            for iw in common_neighbors
                        )
                        h_second = sum_common_weight
                        h_second1 = (h_second + self.W1[id1, id2]) / (len(common_neighbors) + 1)
                        
                        self.Ws[id1, id2] = h_second1
                        self.Ws[id2, id1] = h_second1
        
        # Filter interactions based on weights
        self.filter_interactions()
    
    def filter_interactions(self) -> None:
        
        # Create a new relations dictionary with filtered interactions
        new_relations = defaultdict(list)
        
        for id1 in self.relations:
            for id2 in self.relations[id1]:
                if self.Ws[id1, id2] > 0.0:
                    new_relations[id1].append(id2)
        
        self.relations = {
            k: v for k, v in new_relations.items() 
            if len(v) > 1
        }
    
    def calculate_structural_similarity(self, id1: int, id2: int) -> float:
       
        neighbors1 = set(self.relations.get(id1, []))
        neighbors2 = set(self.relations.get(id2, []))
        
        # Include the proteins themselves in their neighborhoods
        neighbors1.add(id1)
        neighbors2.add(id2)
        
        common_neighbors = neighbors1 & neighbors2
        denominator = math.sqrt(len(neighbors1) * len(neighbors2))
        
        return len(common_neighbors) / denominator if denominator > 0 else 0.0
    
    def detect_core_complexes(self) -> Dict[int, List[int]]:
        
        core_complexes = {}
        
        for id1 in self.relations:
            complex_members = {id1}
            
            for id2 in self.relations[id1]:
                if id1 < id2:
                    similarity = self.calculate_structural_similarity(id1, id2)
                    if similarity > self.structural_similarity_threshold:
                        complex_members.add(id2)
            
            if len(complex_members) >= 2:
                core_complexes[id1] = list(complex_members)
        
        return core_complexes
    
    def find_attachments(self, core_complexes: Dict[int, List[int]]) -> Dict[int, List[int]]:
    
        complexes = {}
        
        for core_id, core_members in core_complexes.items():
            # Find all neighboring proteins not in the core
            attachment_candidates = set()
            for protein in core_members:
                attachment_candidates.update(
                    p for p in self.relations.get(protein, []) 
                    if p not in core_members
                )
            
            if not attachment_candidates:
                complexes[core_id] = core_members
                continue
            
            # Calculate average edge weight within the core
            in_edges, _ = self.clustering_coefficient(core_members)
            avg_core_edges = 2 * in_edges / len(core_members)
            
            # Classify attachment candidates
            local_attachments = set()
            overlapping_attachments = set()
            
            for candidate in attachment_candidates:
                in_weight = 0.0
                out_weight = 0.0
                count = 0
                
                for neighbor in self.relations.get(candidate, []):
                    if neighbor in core_members:
                        in_weight += self.Ws[candidate, neighbor]
                        count += 1
                    else:
                        out_weight += self.Ws[candidate, neighbor]
                
                if count >= 2:
                    if in_weight <= out_weight:
                        if in_weight >= 0.5 * avg_core_edges:
                            overlapping_attachments.add(candidate)
                    else:
                        local_attachments.add(candidate)
            
            # Combine all attachments
            all_attachments = local_attachments | overlapping_attachments
            complexes[core_id] = core_members + list(all_attachments)
        
        return complexes
    
    def clustering_coefficient(self, proteins: List[int]) -> Tuple[float, int]:
        """
        Calculate clustering coefficient for a set of proteins.
        Returns (sum_of_in_edges, count_of_edges)
        """
        protein_set = set(proteins)
        sum_edges = 0.0
        count = 0
        
        for i, id1 in enumerate(proteins):
            for id2 in proteins[i+1:]:
                if id2 in self.relations.get(id1, []):
                    sum_edges += self.Ws[id1, id2]
                    count += 1
        
        return sum_edges, count
    
    def overlap_score(self, set1: Set[int], set2: Set[int]) -> float:
        
        intersection = set1 & set2
        union = set1 | set2
        return len(intersection) / len(union) if union else 0.0
    
    def filter_redundant_complexes(self, complexes: Dict[int, List[int]], 
                                 overlap_threshold: float = 1.0) -> Dict[int, List[int]]:
    
        visited = set()
        result = {}
        complex_items = list(complexes.items())
        
        for i, (id1, members1) in enumerate(complex_items):
            if id1 in visited:
                continue
                
            current_members = set(members1)
            
            for j, (id2, members2) in enumerate(complex_items[i+1:], i+1):
                if id2 in visited:
                    continue
                    
                score = self.overlap_score(current_members, set(members2))
                if score >= overlap_threshold:
                    visited.add(id2)
                    current_members.update(members2)
            
            result[id1] = list(current_members)
        
        return result
    
    def save_results(self, complexes: Dict[int, List[int]], output_file: str) -> None:
        
        with open(output_file, 'w') as f:
            for complex_id, members in complexes.items():
                if len(members) >= 3:  # Only save complexes with at least 3 proteins
                    proteins = [self.id_to_protein[pid] for pid in members]
                    line = f"{complex_id}\t{' '.join(proteins)}\n"
                    f.write(line)
    
    def run(self, output_file: str) -> None:
        
        start_time = time.time()
        
        print("Loading interactions...")
        self.load_interactions()
        
        print("Calculating Jaccard distances...")
        self.calculate_jaccard_distance()
        
        print("Calculating ECV2 weights...")
        self.calculate_ecv2_weights()
        
        print("Detecting core complexes...")
        core_complexes = self.detect_core_complexes()
        
        print("Finding attachment proteins...")
        complexes = self.find_attachments(core_complexes)
        
        print("Filtering redundant complexes...")
        filtered_complexes = self.filter_redundant_complexes(complexes)
        
        print("Saving results...")
        self.save_results(filtered_complexes, output_file)
        
        elapsed = time.time() - start_time
        print(f"EWCA completed in {elapsed:.2f} seconds")
        print(f"Results saved to {output_file}")


# Example usage
if __name__ == "__main__":
    # Initialize with your interaction file and desired parameters
    ewca = EWCA(
        interaction_file="/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_STRING_humain.txt",
        structural_similarity_threshold=0.4
    )
    
    # Run the algorithm
    ewca.run("/Users/ryham/Documents/mémoireM2/Master_final_project/Data/results/tmp/STRING_humain_complexes_ewca2.txt")

Loading interactions...
Total number of proteins: 12397
Total number of interactions: 101652
Calculating Jaccard distances...
Calculating ECV2 weights...
Detecting core complexes...
Finding attachment proteins...
Filtering redundant complexes...
Saving results...
EWCA completed in 69.15 seconds
Results saved to /Users/ryham/Documents/mémoireM2/Master_final_project/Data/results/tmp/STRING_humain_complexes_ewca2.txt


SE-DMTG

In [14]:
import random
from collections import defaultdict
from typing import Dict, List, Tuple, Set
import math
from dataclasses import dataclass
from tqdm import tqdm  # Pour la barre de progression

@dataclass
class ProteinNetwork:
    """Represents a protein-protein interaction network"""
    relations: Dict[str, List[str]]  # Protein ID to list of neighbors
    weights: Dict[Tuple[str, str], float]  # Edge weights
    
    @classmethod
    def from_weighted_network(cls, network_file: str, has_header: bool = True):
        """Create network from weighted interaction file"""
        relations = defaultdict(list)
        weights = {}
        
        # Ajout de la barre de progression pour le chargement
        print("Loading network data...")
        with open(network_file) as f:
            # Skip header if present
            if has_header:
                next(f)
            
            # Compter le nombre de lignes pour la barre de progression
            lines = list(f)
            for line in tqdm(lines, desc="Processing interactions"):
                if line.strip():
                    parts = line.strip().split()
                    # Ensure we have exactly 3 columns
                    if len(parts) != 3:
                        continue
                    
                    try:
                        protein1, protein2, weight = parts[0], parts[1], float(parts[2])
                    except (ValueError, IndexError):
                        continue
                    
                    # Add to relations
                    relations[protein1].append(protein2)
                    relations[protein2].append(protein1)
                    
                    # Add weights (undirected)
                    weights[(protein1, protein2)] = weight
                    weights[(protein2, protein1)] = weight
        
        return cls(relations=dict(relations), weights=weights)


class SEDMTG:
    """Implementation of the SEDMTG algorithm for protein complex detection"""
    
    def __init__(self, network: ProteinNetwork, iterations: int = 10):
        self.network = network
        self.iterations = iterations
    
    def find_seeds(self) -> Dict[str, float]:
        """Step 1: Generate seed queue based on node scores"""
        node_scores = {}
        proteins = list(self.network.relations.keys())
        
        # Ajout de la barre de progression
        for protein in tqdm(proteins, desc="Finding seeds"):
            neighbors = self.network.relations.get(protein, [])
            if len(neighbors) >= 2:
                degree_weight = sum(self.network.weights.get((protein, neighbor), 0) 
                                  for neighbor in neighbors)
                
                subgraph = neighbors + [protein]
                density, _ = self._calculate_density(subgraph)
                score = density * degree_weight
                node_scores[protein] = score
        
        # Shuffle seeds to introduce randomness
        seeds = list(node_scores.keys())
        random.shuffle(seeds)
        
        return {protein: node_scores[protein] for protein in seeds}
    
    def detect_complexes(self) -> Dict[int, List[str]]:
        """Main method to detect protein complexes"""
        all_complexes = defaultdict(list)
        
        # Barre de progression pour les itérations principales
        for _ in tqdm(range(self.iterations), desc="Detecting complexes", total=self.iterations):
            seeds = self.find_seeds()
            seed_proteins = list(seeds.keys())
            
            # Randomly sample seeds
            sampled_seeds = random.sample(seed_proteins, len(seed_proteins))
            seed_dict = {i: protein for i, protein in enumerate(sampled_seeds)}
            
            # Detect complexes from these seeds
            complexes, _, _, _ = self._detect_complexes_from_seeds(seed_dict)
            
            # Filter redundant complexes
            filtered_complexes = self._filter_redundant_complexes(complexes)
            
            # Store results
            for complex_id, proteins in filtered_complexes.items():
                all_complexes[len(all_complexes)] = proteins
        
        return dict(all_complexes)
    
    def save_complexes_to_file(self, complexes: Dict[int, List[str]], output_file: str):
        """Save detected complexes to a file with the required format"""
        with open(output_file, 'w') as f:
            # Tri des complexes par taille (décroissant) comme dans votre exemple
            sorted_complexes = sorted(complexes.items(), key=lambda x: len(x[1]), reverse=True)
            
            # Réindexation pour avoir des IDs séquentiels
            for new_id, (old_id, proteins) in enumerate(sorted_complexes, 1):
                # Format: ID suivi d'une tabulation puis les protéines séparées par des espaces
                f.write(f"{new_id}\t{' '.join(proteins)}\n")
    
    def _detect_complexes_from_seeds(self, seeds: Dict[int, str]) -> Tuple[Dict[int, List[str]], int, Dict[int, float], float]:
        """Step 2-3: Form initial clusters and extend/correct them"""
        complexes = defaultdict(list)
        complex_scores = {}
        total_iterations = 0
        visited = set()
        
        # Calculate average score threshold
        avg_score = self._calculate_average_seed_score(seeds)
        
        for seed_id, protein in seeds.items():
            if protein not in visited:
                neighbors = self.network.relations.get(protein, [])
                initial_cluster = neighbors + [protein]
                
                # Only proceed if cluster meets basic criteria
                if len(set(initial_cluster)) >= 3:
                    score, _, _ = self._composite_score(initial_cluster)
                    
                    if score >= avg_score:
                        # Form initial cluster
                        initial_graph = [protein]
                        
                        # Extend and refine cluster
                        final_cluster, iterations = self._refine_cluster(initial_graph)
                        total_iterations += iterations
                        
                        # Check final cluster quality
                        final_score, _, _ = self._composite_score(final_cluster)
                        if final_score > avg_score and len(final_cluster) >= 3:
                            complexes[len(complexes)] = sorted(list(set(final_cluster)))
                            complex_scores[len(complexes)] = final_score
                            visited.update(final_cluster)
        
        avg_iterations = total_iterations / len(complexes) if complexes else 0
        return complexes, len(complexes), complex_scores, avg_iterations
    
    def _refine_cluster(self, initial_cluster: List[str], max_iterations: int = 100) -> Tuple[List[str], int]:
        """Step 3: Extend and correct the cluster iteratively"""
        current_cluster = initial_cluster.copy()
        iterations = 0
        
        while iterations < max_iterations:
            iterations += 1
            old_cluster = current_cluster.copy()
            
            # Extension phase
            current_cluster = self._extend_cluster(current_cluster)
            
            # Correction phase
            current_cluster = self._correct_cluster(current_cluster)
            
            # Check for convergence
            if set(old_cluster) == set(current_cluster):
                break
        
        return list(set(current_cluster)), iterations
    
    def _extend_cluster(self, cluster: List[str]) -> List[str]:
        """Add relevant proteins to the cluster"""
        neighbors = self._get_cluster_neighbors(cluster)
        extended_cluster = cluster.copy()
        
        while neighbors:
            neighbors = list(set(neighbors))
            current_score, sum_weight, _ = self._composite_score(extended_cluster)
            
            # Find best node to add
            best_node, best_score = self._find_best_addition(extended_cluster, neighbors, sum_weight)
            
            # Calculate addition criteria
            common_neighbors = len(set(self.network.relations.get(best_node, [])) & set(extended_cluster))
            threshold = current_score * len(extended_cluster)
            
            # Check if addition improves the cluster
            if best_score > current_score and common_neighbors >= threshold:
                extended_cluster.append(best_node)
                neighbors.remove(best_node)
                neighbors.extend(self._get_new_neighbors(extended_cluster, best_node))
            else:
                break
        
        return extended_cluster
    
    def _correct_cluster(self, cluster: List[str]) -> List[str]:
        """Remove irrelevant proteins from the cluster"""
        if len(cluster) <= 2:
            return cluster.copy()
        
        corrected_cluster = cluster.copy()
        removable = self._get_removable_nodes(corrected_cluster)
        
        while removable:
            current_score, sum_weight, _ = self._composite_score(corrected_cluster)
            
            # Find worst node to remove
            worst_node, worst_score = self._find_worst_removal(corrected_cluster, removable, sum_weight)
            
            # Calculate removal criteria
            common_neighbors = len(set(self.network.relations.get(worst_node, [])) & set(corrected_cluster))
            threshold = current_score * len(corrected_cluster)
            
            # Check if removal improves the cluster
            if worst_score > current_score and common_neighbors <= threshold:
                corrected_cluster.remove(worst_node)
                removable.remove(worst_node)
            else:
                break
            
            if len(corrected_cluster) <= 2:
                break
        
        return corrected_cluster
    
    def _filter_redundant_complexes(self, complexes: Dict[int, List[str]], overlap_threshold: float = 0.8) -> Dict[int, List[str]]:
        """Step 4: Filter redundant protein complexes"""
        # Sort complexes by size (descending)
        sorted_complexes = sorted(complexes.items(), key=lambda x: len(x[1]), reverse=True)
        
        filtered = {}
        excluded = set()
        
        for i, (idx1, complex1) in enumerate(sorted_complexes):
            if idx1 not in excluded:
                filtered[idx1] = complex1
                
                # Compare with remaining complexes
                for j in range(i + 1, len(sorted_complexes)):
                    idx2, complex2 = sorted_complexes[j]
                    if idx2 not in excluded:
                        overlap = self._calculate_overlap(set(complex1), set(complex2))
                        if overlap >= overlap_threshold:
                            excluded.add(idx2)
        
        # Reindex filtered complexes
        return {i: proteins for i, (_, proteins) in enumerate(filtered.items())}
    
    # Helper methods for calculations
    def _calculate_density(self, subgraph: List[str]) -> Tuple[float, float]:
        """Calculate subgraph density and total weight"""
        if len(subgraph) <= 2:
            return 0.0, 0.0
        
        total_weight = 0.0
        nodes = list(set(subgraph))
        
        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                protein1, protein2 = nodes[i], nodes[j]
                total_weight += self.network.weights.get((protein1, protein2), 0)
        
        density = 2 * total_weight / (len(nodes) * (len(nodes) - 1))
        return density, total_weight
    
    def _calculate_modularity(self, subgraph: List[str]) -> Tuple[float, float]:
        """Calculate subgraph modularity and external weight"""
        nodes = list(set(subgraph))
        internal_weight = 0.0
        external_weight = 0.0
        
        # Calculate internal weight
        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                protein1, protein2 = nodes[i], nodes[j]
                internal_weight += self.network.weights.get((protein1, protein2), 0)
        
        # Calculate external weight
        for protein in nodes:
            for neighbor in self.network.relations.get(protein, []):
                if neighbor not in nodes:
                    external_weight += self.network.weights.get((protein, neighbor), 0)
        
        total_weight = internal_weight + external_weight
        modularity = internal_weight / total_weight if total_weight > 0 else 0.0
        return modularity, external_weight
    
    def _composite_score(self, subgraph: List[str]) -> Tuple[float, float, float]:
        """Combined score for subgraph evaluation"""
        density, sum_weight = self._calculate_density(subgraph)
        modularity, sum_out_weight = self._calculate_modularity(subgraph)
        
        if density == 0 or modularity == 0:
            score = (modularity + density) / 2
        else:
            score = (modularity + density + math.sqrt(density * modularity)) / 3
        
        return score, sum_weight, sum_out_weight
    
    def _calculate_average_seed_score(self, seeds: Dict[int, str]) -> float:
        """Calculate average score of seed proteins"""
        total_score = 0.0
        count = 0
        
        for seed_id, protein in seeds.items():
            neighbors = self.network.relations.get(protein, [])
            if len(neighbors) >= 2:
                score, _, _ = self._composite_score(neighbors + [protein])
                total_score += score
                count += 1
        
        return total_score / count if count > 0 else 0.0
    
    def _get_cluster_neighbors(self, cluster: List[str]) -> List[str]:
        """Get neighbors of a cluster not already in it"""
        neighbors = []
        for protein in cluster:
            for neighbor in self.network.relations.get(protein, []):
                if neighbor not in cluster:
                    neighbors.append(neighbor)
        return list(set(neighbors))
    
    def _get_removable_nodes(self, cluster: List[str]) -> List[str]:
        """Get nodes that are candidates for removal"""
        removable = []
        cluster_set = set(cluster)
        
        for protein in cluster:
            neighbors = set(self.network.relations.get(protein, []))
            if not neighbors.issubset(cluster_set):
                removable.append(protein)
        
        return list(set(removable))
    
    def _find_best_addition(self, cluster: List[str], candidates: List[str], current_weight: float) -> Tuple[str, float]:
        """Find the best node to add to the cluster"""
        best_node = None
        best_score = -1
        
        for node in candidates:
            # Calculate new weight if node were added
            new_weight = current_weight
            for protein in cluster:
                new_weight += self.network.weights.get((node, protein), 0)
            
            # Calculate score (using simplified formula)
            score = 2 * new_weight / (len(cluster) + 1)
            
            if score > best_score:
                best_score = score
                best_node = node
        
        return best_node, best_score
    
    def _find_worst_removal(self, cluster: List[str], candidates: List[str], current_weight: float) -> Tuple[str, float]:
        """Find the worst node to remove from the cluster"""
        worst_node = None
        worst_score = float('inf')
        
        for node in candidates:
            # Calculate new weight if node were removed
            new_weight = current_weight
            for protein in cluster:
                if protein != node:
                    new_weight -= self.network.weights.get((node, protein), 0)
            
            # Calculate score (using simplified formula)
            score = 2 * new_weight / (len(cluster))
            
            if score < worst_score:
                worst_score = score
                worst_node = node
        
        return worst_node, worst_score
    
    def _get_new_neighbors(self, cluster: List[str], new_node: str) -> List[str]:
        """Get new neighbors after adding a node to the cluster"""
        new_neighbors = []
        for neighbor in self.network.relations.get(new_node, []):
            if neighbor not in cluster:
                new_neighbors.append(neighbor)
        return new_neighbors
    
    def _calculate_overlap(self, set1: Set[str], set2: Set[str]) -> float:
        """Calculate overlap score between two protein sets"""
        intersection = len(set1 & set2)
        return (intersection * intersection) / (len(set1) * len(set2))


if __name__ == "__main__":
    # Load network data
    network_file = "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/tmp/GO_weighted_STRING_humain.txt"
    output_file = "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/results/tmp/STRING_humain_complexes_sedmtg_v2.txt"
    
    # Create network (assuming the file has a header)
    print("Initializing network...")
    network = ProteinNetwork.from_weighted_network(network_file, has_header=True)
    
    # Run SEDMTG algorithm
    print("Running SEDMTG algorithm...")
    sedmtg = SEDMTG(network, iterations=10)
    protein_complexes = sedmtg.detect_complexes()
    
    # Save results to file
    print("Saving results...")
    sedmtg.save_complexes_to_file(protein_complexes, output_file)
    
    # Print summary
    print("\nProcessing complete!")
    print(f"Detected {len(protein_complexes)} protein complexes")
    print(f"Results saved to: {output_file}")

Initializing network...
Loading network data...


Processing interactions: 100%|██████████| 101651/101651 [00:00<00:00, 359048.65it/s]


Running SEDMTG algorithm...


Detecting complexes: 100%|██████████| 10/10 [37:53<00:00, 227.34s/it]

Saving results...

Processing complete!
Detected 5537 protein complexes
Results saved to: /Users/ryham/Documents/mémoireM2/Master_final_project/Data/results/tmp/STRING_humain_complexes_sedmtg_v2.txt


MPC-C

In [ ]:
import sys
from math import sqrt
import numpy as np
from collections import defaultdict
import copy
from numpy.matlib import zeros

class MPCC:
    def __init__(self):
        self.label_id = {}
        self.id_label = {}
        self.relations = defaultdict(list)
        self.weights = None
        
    def load_interactions(self, ppi_file):
        """Load PPI data with pre-calculated weights"""
        index = 0
        protein_list = []
        relations = defaultdict(list)
        weights = {}
        
        with open(ppi_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 3:  # Expecting protein1, protein2, weight
                    p1, p2, weight = parts[0], parts[1], float(parts[2])
                    
                    # Create label-id mappings if needed
                    if p1 not in self.label_id:
                        self.label_id[p1] = index
                        self.id_label[index] = p1
                        protein_list.append(p1)
                        index += 1
                    if p2 not in self.label_id:
                        self.label_id[p2] = index
                        self.id_label[index] = p2
                        protein_list.append(p2)
                        index += 1
                    
                    # Add relations and weights
                    id1, id2 = self.label_id[p1], self.label_id[p2]
                    if id2 not in relations[id1] and id1 != id2:
                        relations[id1].append(id2)
                        relations[id2].append(id1)
                        weights[(id1, id2)] = weight
                        weights[(id2, id1)] = weight
        
        self.Protein_num = index
        self.relations = relations
        self.weights = weights
        return self.label_id, self.id_label, self.Protein_num, self.relations, protein_list
    
    def remove_false_positives(self):
        """Remove interactions with zero weights"""
        for id in self.relations:
            neighbors = self.relations[id]
            new_neighbors = [it for it in neighbors if self.weights.get((id, it), 0) > 0]
            self.relations[id] = new_neighbors
        return self.relations
    
    def read_gene_expression(self, gene_file):
        """Read gene expression data (for active time calculation)"""
        gene_expression_data = defaultdict(list)
        with open(gene_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if parts[0] in self.label_id:
                    # Assuming 12 time points with 3 replicates each
                    expr_values = [float(x) for x in parts[1:37]]  # 36 values (12x3)
                    # Average replicates
                    avg_expr = [(expr_values[i] + expr_values[i+12] + expr_values[i+24])/3 
                               for i in range(12)]
                    gene_expression_data[parts[0]] = avg_expr
        return gene_expression_data
    
    def proteins_active_time(self, gene_expression_data):
        """Calculate active time probabilities for proteins"""
        Time_num = 12
        Time_protein_dic = [{} for _ in range(Time_num)]
        
        for protein in gene_expression_data:
            if protein in self.label_id:
                expr = gene_expression_data[protein]
                Temp_mean = np.mean(expr)
                Temp_sd = np.std(expr, ddof=1)
                
                # Calculate thresholds
                thresh_3SD = Temp_mean + 3 * sqrt(Temp_sd) * (Temp_sd / (1 + Temp_sd))
                thresh_2SD = Temp_mean + 2 * sqrt(Temp_sd) * (Temp_sd / (1 + Temp_sd))
                thresh_1SD = Temp_mean + 1 * sqrt(Temp_sd) * (Temp_sd / (1 + Temp_sd))
                
                # Assign probabilities based on expression levels
                protein_id = self.label_id[protein]
                for j in range(Time_num):
                    val = expr[j]
                    if val >= thresh_3SD:
                        Time_protein_dic[j][protein_id] = 0.9973
                    elif val >= thresh_2SD:
                        Time_protein_dic[j][protein_id] = 0.9545
                    elif val >= thresh_1SD:
                        Time_protein_dic[j][protein_id] = 0.6827
                    else:
                        Time_protein_dic[j][protein_id] = 0.0
                        
        return Time_protein_dic
    
    def construct_subnetworks(self, Time_protein_dic):
        """Construct dynamic subnetworks based on active times"""
        dict_networks = {}
        dict_networks_Weight = {}
        
        for time_idx, active_proteins in enumerate(Time_protein_dic, 1):
            relation = defaultdict(list)
            N = len(self.id_label)
            weight_id = zeros((N, N))
            
            # Filter out inactive proteins
            active_proteins = {k:v for k,v in active_proteins.items() if v > 0}
            
            for protein_id in active_proteins:
                neighbors = self.relations[protein_id]
                active_neighbors = []
                protein_stage = active_proteins[protein_id]
                
                for neighbor_id in neighbors:
                    if neighbor_id in active_proteins:
                        neighbor_stage = active_proteins[neighbor_id]
                        # Weight is product of active probabilities
                        w = protein_stage * neighbor_stage
                        weight_id[protein_id, neighbor_id] = w
                        weight_id[neighbor_id, protein_id] = w
                        active_neighbors.append(neighbor_id)
                
                if len(active_neighbors) >= 2:
                    relation[protein_id] = active_neighbors
            
            dict_networks[time_idx] = relation
            dict_networks_Weight[time_idx] = weight_id
            
        return dict_networks, dict_networks_Weight
    
    def calculate_topology_scores(self, network, N):
        """Calculate topology similarity scores"""
        Topology_weight = zeros((N, N))
        for id in network:
            neighbors = network[id]
            for it in neighbors:
                neighbors1 = network[it]
                score = self.TO_ij_score(neighbors, neighbors1)
                Topology_weight[id, it] = score
                Topology_weight[it, id] = score
        return Topology_weight
    
    def TO_ij_score(self, list1, list2):
        """Calculate topological overlap score"""
        common = set(list1) & set(list2)
        top_sum = len(common)
        top_down = min(len(list1), len(list2))
        
        if top_down > 0:
            score = (top_sum/sqrt(len(list1)*len(list2)) + 
                     top_sum/top_down + 
                     2*top_sum/(len(list1)+len(list2))) / 3
        else:
            score = 0.0
        return score
    
    def detect_seeds(self, id_list, network, weight_network):
        """Detect seed proteins for complex formation"""
        seeds = defaultdict(list)
        Graph_avgdensity = 0.0
        num = 0
        
        # Calculate average graph density
        for id in id_list:
            neighbors = network.get(id, [])
            if id not in neighbors:
                neighbors.append(id)
            if len(neighbors) >= 2:
                density, _ = self.graph_density(neighbors, network, weight_network)
                Graph_avgdensity += density
                num += 1
                
        if num > 0:
            Graph_avgdensity /= num
        
        # Identify seed proteins
        visit_node = []
        for id1 in id_list:
            if id1 not in visit_node:
                neighbors1 = network.get(id1, [])
                if id1 not in neighbors1:
                    neighbors1.append(id1)
                
                if len(neighbors1) >= 2:
                    # Find core proteins
                    protein_cores = [id1]
                    density1, weight_sum1 = self.graph_density(neighbors1, network, weight_network)
                    avg_deg = 2 * weight_sum1 / len(neighbors1)
                    
                    for ih in neighbors1:
                        neighbor_ih = network.get(ih, [])
                        sum_w = sum(weight_network[ih,it] for it in neighbor_ih 
                                   if it in neighbors1)
                        if sum_w >= avg_deg:
                            protein_cores.append(ih)
                    
                    # Find essential proteins connected to core
                    left_proteins = list(set(neighbors1) - set(protein_cores))
                    essential_cores = []
                    for ib in left_proteins:
                        neighbors_ib = network.get(ib, [])
                        common = set(neighbors_ib) & set(protein_cores)
                        if len(common) >= 2 and len(common) > 0.5*len(protein_cores):
                            essential_cores.append(ib)
                    
                    protein_cores = list(set(essential_cores) | set(protein_cores))
                    density2, _ = self.graph_density(protein_cores, network, weight_network)
                    
                    if density2 > Graph_avgdensity:
                        seeds[id1] = protein_cores
                        visit_node.extend(protein_cores)
        
        return seeds
    
    def graph_density(self, graph, network, weight):
        """Calculate graph density"""
        weight_sum = 0.0
        if len(graph) >= 2:
            for id in graph:
                neighbors = network.get(id, [])
                for it in neighbors:
                    if id < it and it in graph:
                        weight_sum += weight[id, it]
            density = 2 * weight_sum / (len(graph) * (len(graph)-1))
        else:
            density = 0.0
        return density, weight_sum
    
    def identify_complexes(self, seeds, network, weight_network):
        """Identify protein complexes from seeds"""
        complexes = defaultdict(list)
        k = 1
        
        for seed_id in seeds:
            initial_graph = seeds[seed_id]
            if len(initial_graph) >= 2:
                # Grow the complex iteratively
                protein_complex, _ = self.grow_complex(initial_graph, network, weight_network, 1)
                protein_complex = list(set(protein_complex))
                
                if len(protein_complex) >= 3:
                    complexes[k] = protein_complex
                    k += 1
                    
        return complexes
    
    def grow_complex(self, initial_graph, relations, weight, iterations):
        """Grow complex iteratively by adding/removing nodes"""
        subgraph_new = self.add_nodes(initial_graph, relations, weight)
        subgraph_new = self.remove_nodes(subgraph_new, relations, weight)
        
        if set(subgraph_new) != set(initial_graph) and iterations < 10:
            return self.grow_complex(subgraph_new, relations, weight, iterations+1)
        else:
            return subgraph_new, iterations
    
    def add_nodes(self, graph, relations, weight):
        """Add nodes to the complex that improve its score"""
        add_nodes = []
        for id in graph:
            neighbors = relations.get(id, [])
            for it in neighbors:
                if it not in graph and it not in add_nodes:
                    add_nodes.append(it)
        
        while add_nodes:
            start_score = self.graph_entropy(graph, relations, weight)
            max_node = self.find_max_node(add_nodes, graph, relations, weight)
            candidate = graph + [max_node]
            new_score = self.graph_entropy(candidate, relations, weight)
            
            if new_score > start_score:
                graph.append(max_node)
                add_nodes.remove(max_node)
            else:
                break
                
        return graph
    
    def remove_nodes(self, graph, relations, weight):
        """Remove nodes from complex that improve its score"""
        if len(graph) <= 2:
            return graph
            
        remove_list = [id for id in graph 
                      if not set(relations.get(id, [])).issubset(graph)]
        
        while remove_list:
            start_score = self.graph_entropy(graph, relations, weight)
            min_node = self.find_min_node(remove_list, graph, relations, weight)
            candidate = [x for x in graph if x != min_node]
            new_score = self.graph_entropy(candidate, relations, weight)
            
            if new_score > start_score:
                graph.remove(min_node)
                remove_list.remove(min_node)
                if len(graph) <= 2:
                    break
            else:
                break
                
        return graph
    
    def find_max_node(self, nodes, graph, relations, weight):
        """Find node with maximum connection weight to current graph"""
        max_node = nodes[0]
        max_weight = sum(weight[max_node, it] for it in relations.get(max_node, []) 
                        if it in graph)
        
        for id in nodes[1:]:
            sum_w = sum(weight[id, it] for it in relations.get(id, []) 
                    if it in graph)
            if sum_w > max_weight:
                max_node, max_weight = id, sum_w
                
        return max_node
    
    def find_min_node(self, nodes, graph, relations, weight):
        """Find node with minimum connection weight to current graph"""
        min_node = nodes[0]
        min_weight = sum(weight[min_node, it] for it in relations.get(min_node, []) 
                    if it in graph)
        
        for id in nodes[1:]:
            sum_w = sum(weight[id, it] for it in relations.get(id, []) 
                    if it in graph)
            if sum_w < min_weight:
                min_node, min_weight = id, sum_w
                
        return min_node
    
    def graph_entropy(self, graph, relations, weight):
        """Calculate graph clustering score"""
        if len(graph) < 2:
            return 0.0
            
        weight_in = 0.0
        weight_out = 0.0
        count_out = 0
        
        for id in graph:
            neighbors = relations.get(id, [])
            inner_sum = sum(weight[id, it] for it in neighbors if it in graph and it > id)
            outer_sum = sum(weight[id, it] for it in neighbors if it not in graph)
            
            weight_in += inner_sum
            if outer_sum > 0:
                weight_out += outer_sum
                count_out += 1
                
        weight_in = 2 * weight_in / len(graph)
        weight_out = weight_out / count_out if count_out > 0 else 0.0
        
        if weight_in + weight_out > 0:
            return weight_in / (weight_in + weight_out)
        return 0.0
    
    def read_essential_proteins(self, essential_file):
        """Read list of essential proteins"""
        essential_proteins = []
        with open(essential_file, "r") as f:
            for line in f:
                protein = line.strip().split()[0]
                if protein in self.label_id:
                    essential_proteins.append(self.label_id[protein])
        return list(set(essential_proteins))
    
    def overlap_score(self, clusterA, clusterB):
        """Calculate overlap score between two clusters"""
        intersect = len(set(clusterA) & set(clusterB))
        return (intersect * intersect) / (len(clusterA) * len(clusterB))
    
    def filter_redundant(self, complexes, threshold=0.8):
        """Filter redundant complexes based on overlap"""
        to_remove = set()
        complex_ids = list(complexes.keys())
        
        for i in range(len(complex_ids)):
            id1 = complex_ids[i]
            if id1 in to_remove:
                continue
                
            for j in range(i+1, len(complex_ids)):
                id2 = complex_ids[j]
                if id2 in to_remove:
                    continue
                    
                score = self.overlap_score(complexes[id1], complexes[id2])
                if score >= threshold:
                    to_remove.add(id2)
        
        return {k:v for k,v in complexes.items() if k not in to_remove}
    
    def sort_complexes(self, complexes, scores=None):
        """Sort complexes by size or score"""
        if scores:
            sorted_ids = sorted(scores.items(), key=lambda x: x[1], reverse=True)
            return {i+1: complexes[k] for i, (k,v) in enumerate(sorted_ids)}
        else:
            sorted_ids = sorted(complexes.items(), key=lambda x: len(x[1]), reverse=True)
            return {i+1: v for i, (k,v) in enumerate(sorted_ids)}
    
    def run(self, ppi_file, gene_file, essential_file, output_file):
        """Main workflow to identify protein complexes"""
        # Step 1: Load pre-calculated interactions
        self.load_interactions(ppi_file)
        
        # Step 2: Remove false positives
        self.remove_false_positives()
        
        # Step 3: Process gene expression data
        gene_data = self.read_gene_expression(gene_file)
        time_protein = self.proteins_active_time(gene_data)
        
        # Step 4: Construct dynamic subnetworks
        subnetworks, subnetwork_weights = self.construct_subnetworks(time_protein)
        
        # Step 5: Read essential proteins
        essential_proteins = self.read_essential_proteins(essential_file)
        
        # Step 6: Visit subnetworks to find complexes
        all_complexes = defaultdict(list)
        complex_scores = {}
        count = 1
        
        for time_idx in subnetworks:
            network = subnetworks[time_idx]
            weights = subnetwork_weights[time_idx]
            N = len(self.id_label)
            
            # Combine weights with topology scores
            topo_weights = self.calculate_topology_scores(network, N)
            combined_weights = zeros((N, N))
            
            for i in range(N):
                for j in range(N):
                    if i < j and weights[i,j] > 0 and topo_weights[i,j] > 0:
                        # Harmonic mean of weights
                        combined_weights[i,j] = 2 * weights[i,j] * topo_weights[i,j] / (weights[i,j] + topo_weights[i,j])
                        combined_weights[j,i] = combined_weights[i,j]
            
            # Clean network by removing low-weight edges
            id_list = list(network.keys())
            for id in id_list:
                neighbors = [it for it in network[id] 
                           if combined_weights[id,it] > 0]
                network[id] = neighbors
            
            # Detect seeds and identify complexes
            seeds = self.detect_seeds(id_list, network, combined_weights)
            complexes = self.identify_complexes(seeds, network, combined_weights)
            
            # Store results
            for cid in complexes:
                if len(complexes[cid]) >= 3:
                    all_complexes[count] = complexes[cid]
                    complex_scores[count] = self.graph_entropy(complexes[cid], network, combined_weights)
                    count += 1
        
        # Step 7: Process static network
        static_complexes, static_scores = self.process_static_network()
        
        # Step 8: Merge results
        merged_complexes, merged_scores = self.merge_results(all_complexes, complex_scores, 
                                                           static_complexes, static_scores)
        
        # Step 9: Filter and sort results
        filtered = self.filter_redundant(merged_complexes)
        sorted_complexes = self.sort_complexes(filtered, merged_scores)
        
        # Step 10: Save results
        self.save_results(sorted_complexes, output_file)
        
        return sorted_complexes
    
    def process_static_network(self):
        """Process the full static PPI network"""
        N = len(self.id_label)
        topo_weights = self.calculate_topology_scores(self.relations, N)
        combined_weights = zeros((N, N))
        
        # Create weight matrix from loaded weights
        weight_matrix = zeros((N, N))
        for (i,j), w in self.weights.items():
            weight_matrix[i,j] = w
        
        # Combine with topology scores
        for i in range(N):
            for j in range(N):
                if i < j and weight_matrix[i,j] > 0 and topo_weights[i,j] > 0:
                    combined_weights[i,j] = 2 * weight_matrix[i,j] * topo_weights[i,j] / (weight_matrix[i,j] + topo_weights[i,j])
                    combined_weights[j,i] = combined_weights[i,j]
        
        # Detect seeds and identify complexes
        seeds = self.detect_seeds(list(self.relations.keys()), self.relations, combined_weights)
        complexes = self.identify_complexes(seeds, self.relations, combined_weights)
        
        # Calculate scores
        complex_scores = {}
        all_complexes = defaultdict(list)
        count = 1
        
        for cid in complexes:
            if len(complexes[cid]) >= 3:
                all_complexes[count] = complexes[cid]
                complex_scores[count] = self.graph_entropy(complexes[cid], self.relations, combined_weights)
                count += 1
                
        return all_complexes, complex_scores
    
    def merge_results(self, dyn_complexes, dyn_scores, static_complexes, static_scores):
        """Merge dynamic and static complexes"""
        merged = defaultdict(list)
        scores = {}
        
        # Add dynamic complexes
        for cid in dyn_complexes:
            merged[cid] = dyn_complexes[cid]
            scores[cid] = dyn_scores[cid]
        
        # Add static complexes with offset IDs
        offset = max(dyn_complexes.keys()) if dyn_complexes else 0
        for cid in static_complexes:
            merged[offset+cid] = static_complexes[cid]
            scores[offset+cid] = static_scores[cid]
            
        return merged, scores
    
    def save_results(self, complexes, output_file):
        """Save complexes to output file"""
        with open(output_file, "w") as f:
            for cid in complexes:
                proteins = [self.id_label[pid] for pid in complexes[cid]]
                line = f"{cid} {' '.join(proteins)}\n"
                f.write(line)

# Example usage
if __name__ == "__main__":
    mpcc = MPCC()
    complexes = mpcc.run(
        ppi_file="protein_interactions.txt",  # Format: protein1 protein2 weight
        gene_file="gene_expression.txt",     # Gene expression data
        essential_file="essential_proteins.txt",  # List of essential proteins
        output_file="predicted_complexes.txt"
    )

In [3]:
import sys
from math import sqrt
import numpy as np
from collections import defaultdict
import copy
from numpy.matlib import zeros

class MPCC:
    def __init__(self):
        self.label_id = {}
        self.id_label = {}
        self.relations = defaultdict(list)
        self.weights = None
        
    def load_interactions(self, ppi_file):
        """Load PPI data with pre-calculated weights (handles header)"""
        index = 0
        protein_list = []
        relations = defaultdict(list)
        weights = {}
        has_header = True  # Modifier à False si le fichier n'a pas d'en-tête
        
        with open(ppi_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                
                # Skip header line if exists
                if has_header and index == 0:
                    has_header = False  # On a traité l'en-tête
                    continue  # Passe à la ligne suivante
                
                if len(parts) >= 3:  # Format attendu : protein1 protein2 weight
                    p1, p2, weight = parts[0], parts[1], float(parts[2])
                    
                    # Gestion des identifiants
                    if p1 not in self.label_id:
                        self.label_id[p1] = index
                        self.id_label[index] = p1
                        protein_list.append(p1)
                        index += 1
                    if p2 not in self.label_id:
                        self.label_id[p2] = index
                        self.id_label[index] = p2
                        protein_list.append(p2)
                        index += 1
                    
                    # Ajout des interactions
                    id1, id2 = self.label_id[p1], self.label_id[p2]
                    if id2 not in relations[id1] and id1 != id2:
                        relations[id1].append(id2)
                        relations[id2].append(id1)
                        weights[(id1, id2)] = weight
                        weights[(id2, id1)] = weight
        
        self.Protein_num = index
        self.relations = relations
        self.weights = weights
        return self.label_id, self.id_label, self.Protein_num, self.relations, protein_list
    
    def remove_false_positives(self):
        """Remove interactions with zero weights"""
        for id in self.relations:
            neighbors = self.relations[id]
            new_neighbors = [it for it in neighbors if self.weights.get((id, it), 0) > 0]
            self.relations[id] = new_neighbors
        return self.relations
    
    def calculate_topology_scores(self, network, N):
        """Calculate topology similarity scores"""
        Topology_weight = zeros((N, N))
        for id in network:
            neighbors = network[id]
            for it in neighbors:
                neighbors1 = network[it]
                score = self.TO_ij_score(neighbors, neighbors1)
                Topology_weight[id, it] = score
                Topology_weight[it, id] = score
        return Topology_weight
    
    def TO_ij_score(self, list1, list2):
        """Calculate topological overlap score"""
        common = set(list1) & set(list2)
        top_sum = len(common)
        top_down = min(len(list1), len(list2))
        
        if top_down > 0:
            score = (top_sum/sqrt(len(list1)*len(list2)) + 
                     top_sum/top_down + 
                     2*top_sum/(len(list1)+len(list2))) / 3
        else:
            score = 0.0
        return score
    
    def detect_seeds(self, id_list, network, weight_network):
        """Detect seed proteins for complex formation"""
        seeds = defaultdict(list)
        Graph_avgdensity = 0.0
        num = 0
        
        # Calculate average graph density
        for id in id_list:
            neighbors = network.get(id, [])
            if id not in neighbors:
                neighbors.append(id)
            if len(neighbors) >= 2:
                density, _ = self.graph_density(neighbors, network, weight_network)
                Graph_avgdensity += density
                num += 1
                
        if num > 0:
            Graph_avgdensity /= num
        
        # Identify seed proteins
        visit_node = []
        for id1 in id_list:
            if id1 not in visit_node:
                neighbors1 = network.get(id1, [])
                if id1 not in neighbors1:
                    neighbors1.append(id1)
                
                if len(neighbors1) >= 2:
                    # Find core proteins
                    protein_cores = [id1]
                    density1, weight_sum1 = self.graph_density(neighbors1, network, weight_network)
                    avg_deg = 2 * weight_sum1 / len(neighbors1)
                    
                    for ih in neighbors1:
                        neighbor_ih = network.get(ih, [])
                        sum_w = sum(weight_network[ih,it] for it in neighbor_ih 
                                   if it in neighbors1)
                        if sum_w >= avg_deg:
                            protein_cores.append(ih)
                    
                    # Find essential proteins connected to core
                    left_proteins = list(set(neighbors1) - set(protein_cores))
                    essential_cores = []
                    for ib in left_proteins:
                        neighbors_ib = network.get(ib, [])
                        common = set(neighbors_ib) & set(protein_cores)
                        if len(common) >= 2 and len(common) > 0.5*len(protein_cores):
                            essential_cores.append(ib)
                    
                    protein_cores = list(set(essential_cores) | set(protein_cores))
                    density2, _ = self.graph_density(protein_cores, network, weight_network)
                    
                    if density2 > Graph_avgdensity:
                        seeds[id1] = protein_cores
                        visit_node.extend(protein_cores)
        
        return seeds
    
    def graph_density(self, graph, network, weight):
        """Calculate graph density"""
        weight_sum = 0.0
        if len(graph) >= 2:
            for id in graph:
                neighbors = network.get(id, [])
                for it in neighbors:
                    if id < it and it in graph:
                        weight_sum += weight[id, it]
            density = 2 * weight_sum / (len(graph) * (len(graph)-1))
        else:
            density = 0.0
        return density, weight_sum
    
    def identify_complexes(self, seeds, network, weight_network):
        """Identify protein complexes from seeds"""
        complexes = defaultdict(list)
        k = 1
        
        for seed_id in seeds:
            initial_graph = seeds[seed_id]
            if len(initial_graph) >= 2:
                # Grow the complex iteratively
                protein_complex, _ = self.grow_complex(initial_graph, network, weight_network, 1)
                protein_complex = list(set(protein_complex))
                
                if len(protein_complex) >= 3:
                    complexes[k] = protein_complex
                    k += 1
                    
        return complexes
    
    def grow_complex(self, initial_graph, relations, weight, iterations):
        """Grow complex iteratively by adding/removing nodes"""
        subgraph_new = self.add_nodes(initial_graph, relations, weight)
        subgraph_new = self.remove_nodes(subgraph_new, relations, weight)
        
        if set(subgraph_new) != set(initial_graph) and iterations < 10:
            return self.grow_complex(subgraph_new, relations, weight, iterations+1)
        else:
            return subgraph_new, iterations
    
    def add_nodes(self, graph, relations, weight):
        """Add nodes to the complex that improve its score"""
        add_nodes = []
        for id in graph:
            neighbors = relations.get(id, [])
            for it in neighbors:
                if it not in graph and it not in add_nodes:
                    add_nodes.append(it)
        
        while add_nodes:
            start_score = self.graph_entropy(graph, relations, weight)
            max_node = self.find_max_node(add_nodes, graph, relations, weight)
            candidate = graph + [max_node]
            new_score = self.graph_entropy(candidate, relations, weight)
            
            if new_score > start_score:
                graph.append(max_node)
                add_nodes.remove(max_node)
            else:
                break
                
        return graph
    
    def remove_nodes(self, graph, relations, weight):
        """Remove nodes from complex that improve its score"""
        if len(graph) <= 2:
            return graph
            
        remove_list = [id for id in graph 
                      if not set(relations.get(id, [])).issubset(graph)]
        
        while remove_list:
            start_score = self.graph_entropy(graph, relations, weight)
            min_node = self.find_min_node(remove_list, graph, relations, weight)
            candidate = [x for x in graph if x != min_node]
            new_score = self.graph_entropy(candidate, relations, weight)
            
            if new_score > start_score:
                graph.remove(min_node)
                remove_list.remove(min_node)
                if len(graph) <= 2:
                    break
            else:
                break
                
        return graph
    
    def find_max_node(self, nodes, graph, relations, weight):
        """Find node with maximum connection weight to current graph"""
        max_node = nodes[0]
        max_weight = sum(weight[max_node, it] for it in relations.get(max_node, []) 
                        if it in graph)
        
        for id in nodes[1:]:
            sum_w = sum(weight[id, it] for it in relations.get(id, []) 
                    if it in graph)
            if sum_w > max_weight:
                max_node, max_weight = id, sum_w
                
        return max_node
    
    def find_min_node(self, nodes, graph, relations, weight):
        """Find node with minimum connection weight to current graph"""
        min_node = nodes[0]
        min_weight = sum(weight[min_node, it] for it in relations.get(min_node, []) 
                    if it in graph)
        
        for id in nodes[1:]:
            sum_w = sum(weight[id, it] for it in relations.get(id, []) 
                    if it in graph)
            if sum_w < min_weight:
                min_node, min_weight = id, sum_w
                
        return min_node
    
    def graph_entropy(self, graph, relations, weight):
        """Calculate graph clustering score"""
        if len(graph) < 2:
            return 0.0
            
        weight_in = 0.0
        weight_out = 0.0
        count_out = 0
        
        for id in graph:
            neighbors = relations.get(id, [])
            inner_sum = sum(weight[id, it] for it in neighbors if it in graph and it > id)
            outer_sum = sum(weight[id, it] for it in neighbors if it not in graph)
            
            weight_in += inner_sum
            if outer_sum > 0:
                weight_out += outer_sum
                count_out += 1
                
        weight_in = 2 * weight_in / len(graph)
        weight_out = weight_out / count_out if count_out > 0 else 0.0
        
        if weight_in + weight_out > 0:
            return weight_in / (weight_in + weight_out)
        return 0.0
    
    def read_essential_proteins(self, essential_file):
        """Read list of essential proteins"""
        essential_proteins = []
        with open(essential_file, "r") as f:
            for line in f:
                protein = line.strip().split()[0]
                if protein in self.label_id:
                    essential_proteins.append(self.label_id[protein])
        return list(set(essential_proteins))
    
    def overlap_score(self, clusterA, clusterB):
        """Calculate overlap score between two clusters"""
        intersect = len(set(clusterA) & set(clusterB))
        return (intersect * intersect) / (len(clusterA) * len(clusterB))
    
    def filter_redundant(self, complexes, threshold=0.8):
        """Filter redundant complexes based on overlap"""
        to_remove = set()
        complex_ids = list(complexes.keys())
        
        for i in range(len(complex_ids)):
            id1 = complex_ids[i]
            if id1 in to_remove:
                continue
                
            for j in range(i+1, len(complex_ids)):
                id2 = complex_ids[j]
                if id2 in to_remove:
                    continue
                    
                score = self.overlap_score(complexes[id1], complexes[id2])
                if score >= threshold:
                    to_remove.add(id2)
        
        return {k:v for k,v in complexes.items() if k not in to_remove}
    
    def sort_complexes(self, complexes, scores=None):
        """Sort complexes by size or score with robust key handling"""
        if scores:
            # Ensure all score keys exist in complexes
            valid_keys = [k for k in scores.keys() if k in complexes]
            sorted_ids = sorted(((k, scores[k]) for k in valid_keys), 
                              key=lambda x: x[1], reverse=True)
            return {i+1: complexes[k] for i, (k,v) in enumerate(sorted_ids)}
        else:
            sorted_ids = sorted(complexes.items(), 
                              key=lambda x: len(x[1]), 
                              reverse=True)
            return {i+1: v for i, (k,v) in enumerate(sorted_ids)}
    
    
    def run_static(self, ppi_file, output_file="resultats_complexes.txt"):
        """Main workflow with improved error handling"""
        try:
            self.load_interactions(ppi_file)
            self.remove_false_positives()
            print(f"Total proteins: {len(self.id_label)}")
            print(f"Total interactions: {sum(len(v) for v in self.relations.values())//2}")
            
            N = len(self.id_label)
            topo_weights = self.calculate_topology_scores(self.relations, N)
            combined_weights = zeros((N, N))
            
            # Create weight matrix
            weight_matrix = zeros((N, N))
            for (i,j), w in self.weights.items():
                weight_matrix[i,j] = w
            
            # Combine weights
            for i in range(N):
                for j in range(N):
                    if i < j and weight_matrix[i,j] > 0 and topo_weights[i,j] > 0:
                        combined_weights[i,j] = 2 * weight_matrix[i,j] * topo_weights[i,j] / (weight_matrix[i,j] + topo_weights[i,j])
                        combined_weights[j,i] = combined_weights[i,j]
            
            # Detect complexes
            seeds = self.detect_seeds(list(self.relations.keys()), self.relations, combined_weights)
            complexes = self.identify_complexes(seeds, self.relations, combined_weights)
            
            # Calculate scores with key consistency check
            complex_scores = {}
            final_complexes = defaultdict(list)
            count = 1
            
            for cid in list(complexes.keys()):  # Explicit conversion to list
                if len(complexes[cid]) >= 3:
                    final_complexes[count] = complexes[cid]
                    complex_scores[count] = self.graph_entropy(complexes[cid], self.relations, combined_weights)
                    count += 1
            print(f"Complexes keys: {set(final_complexes.keys())}")
            print(f"Scores keys: {set(complex_scores.keys())}")
            missing_keys = set(complex_scores.keys()) - set(final_complexes.keys())
            if missing_keys:
                print(f"WARNING: {len(missing_keys)} score keys missing from complexes")
            # Filter and sort
            filtered = self.filter_redundant(final_complexes)
            sorted_complexes = self.sort_complexes(filtered, complex_scores)
            
            # Save results
            with open(output_file, "w") as f:
                for cid in sorted_complexes:
                    proteins = [self.id_label.get(pid, "UNKNOWN") for pid in sorted_complexes[cid]]  # Safer protein name lookup
                    line = f"{cid} {' '.join(proteins)}\n"
                    f.write(line)
            
            return sorted_complexes
            
        except Exception as e:
            print(f"Error processing network: {str(e)}")
            raise

# Example usage with error handling
if __name__ == "__main__":
    try:
        mpcc = MPCC()
        complexes = mpcc.run_static(
            ppi_file="/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_humain.txt",
            output_file="/Users/ryham/Documents/mémoireM2/Master_final_project/Data/results/tmp/BIOGRID_humain_complexes_mpcc.txt"
        )
        print(f"Successfully identified {len(complexes)} protein complexes")
    except Exception as e:
        print(f"Failed to process: {str(e)}")

Total proteins: 11120
Total interactions: 86981
Complexes keys: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209